# Phase 2 — manual capture, latency and clean-process smoke
## Goal
Both actual models and the consented image diagnostic now run outside Jupyter. Component latency was measured on the passing image with 20 warmups and 100 repetitions. Full calibrated-verification latency and webcam remain NOT_RUN. No camera activates during Run All.

## Setup

In [ ]:
from pathlib import Path
import sys, json, os, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/phase2_verification.yaml').exists())
sys.path.insert(0,str(ROOT/'src'))
os.environ['MPLCONFIGDIR'] = str(ROOT/'.cache/matplotlib')
from face_verification import *
config = load_config(root=ROOT)
REPORTS = ROOT/'reports/phase2'
# Real faces are opt-in. Set these manually only after confirming each participant's consent.
RUN_REAL_FACES = False
ADULT_CONSENT = False
IDENTITY_CONSENT = False
PRIVATE_MANIFEST = ROOT/'data/private/consent_manifest.json'
print('Interpreter:',sys.executable)
print('Real-face workflow enabled:',RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT)


PRIVATE_INPUTS = Path(os.environ.get('FACEATTEND_PRIVATE_INPUTS', str(Path.home()/'AppData/Local/FaceAttend/phase2/private')))
# Explicit executor opt-in; ordinary Run All keeps all defaults false.
if os.environ.get('FACEATTEND_CONSENTED_SMOKE') == '1':
    supplied = json.loads((PRIVATE_INPUTS/'input_manifest.json').read_text())
    assert supplied['adult_identity_verification_consent_confirmed_by_user'] is True
    assert supplied['research_terms_confirmed_by_user'] is True
    RUN_REAL_FACES = ADULT_CONSENT = IDENTITY_CONSENT = True
print('Consented three-image smoke enabled:',RUN_REAL_FACES)


## Fresh-process import/model smoke

In [ ]:
completed = subprocess.run([sys.executable,str(ROOT/'scripts/phase2_verify.py'),'--smoke-no-data'],cwd=ROOT,capture_output=True,text=True,timeout=120)
print(completed.stdout)
assert completed.returncode == 0, completed.stderr
print(json.loads((REPORTS/'latency_report.json').read_text()))

## Consented images from a clean interpreter
The private files remain outside the project; output contains diagnostic metadata only.

In [ ]:
if RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT:
    command = [sys.executable,str(ROOT/'scripts/phase2_private_image_smoke.py'),'--confirm-consented-adult-data','--private-dir',str(PRIVATE_INPUTS),'--skip-latency']
    completed = subprocess.run(command,cwd=ROOT,capture_output=True,text=True,timeout=180)
    assert completed.returncode == 0, 'Private clean-process diagnostic failed; inspect the local error.'
    diagnostic = json.loads(completed.stdout)
    print(json.dumps({'status':diagnostic['status'],'results':diagnostic['results'],'profiles':diagnostic['profiles'],'embeddings_persisted':diagnostic['embeddings_persisted']},indent=2))
else:
    print('Private clean-process image diagnostic opt-in disabled.')

## User-initiated camera capture — manual cell
Confirm adult consent, set `START_ONE_CAPTURE=True`, then run this cell yourself. A visible preview shows **SPACE to capture once, ESC/Q to stop**. Closing the window or the 60-second timeout stops capture. Nothing is saved. Keep unconsenting people out of the frame. No liveness assurance.

If OpenCV preview is unavailable, use a fresh consented local image. Codex has not opened the camera.

In [ ]:
START_ONE_CAPTURE = False
captured = None
if START_ONE_CAPTURE and RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT:
    from face_verification.webcam import capture_one_frame
    captured = capture_one_frame(adult_consent=True,identity_consent=True,user_initiated=True)
    print('Capture result:', 'one in-memory frame' if captured is not None else 'stopped without capture')
else:
    print('WEBCAM_NOT_RUN: requires a deliberate participant action.')

## Full verification benchmark — manual opt-in
Use a valid consented fresh probe and previously enrolled claimed pseudonym after real calibration. Warmup 20, measured repetitions 100. Report mean/median/p95 for SCRFD, embedding from aligned crop, and full verification. CPU image decoding/capture are excluded; preprocessing, transfers and postprocessing are included as documented. Each synchronous `session.run` returns CPU output after GPU work completes.

In [ ]:
RUN_LATENCY = False
CLAIMED_ID = 'volunteer_001'
PROBE_PATH = ROOT/'data/private/fresh_probe.jpg'
if RUN_LATENCY and RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT:
    import cv2
    from face_verification.benchmark import benchmark_verification
    models = initialize_models(config)
    store = TemplateStore(config.path(config.store_dir),key_env=config.key_env,retention_days=config.retention_days)
    probe = captured if captured is not None else cv2.imread(str(PROBE_PATH))
    latency = benchmark_verification(probe,CLAIMED_ID,store,models,adult_consent=True,identity_consent=True)
    (REPORTS/'latency_report.json').write_text(json.dumps(latency,indent=2))
    print(latency)
else:
    print('LATENCY_NOT_RUN: no valid consented probe, enrollment and calibrated threshold; no measurements invented.')

## Clean interpreter with private real data — manual opt-in
The encryption key must already be in the environment. This command loads its own models/templates and does not depend on variables in earlier notebooks. Keep CLI stdout private.

In [ ]:
RUN_FRESH_REAL_VERIFY = False
if RUN_FRESH_REAL_VERIFY and RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT:
    result = subprocess.run([sys.executable,str(ROOT/'scripts/phase2_verify.py'),'--user-id',CLAIMED_ID,
        '--image',str(PROBE_PATH),'--confirm-adult-verification-consent'],cwd=ROOT,capture_output=True,text=True,timeout=120)
    decision = json.loads(result.stdout)
    print({'status':decision['status'],'verified':decision.get('verified',False)})
else:
    print('REAL_FRESH_PROCESS_VERIFY_NOT_RUN: consented inputs and calibration pending.')

## Checks and next steps
SCRFD and ArcFace computational profiles are CUDA-only. Median times: detector 16.38 ms, aligned-crop embedding 5.83 ms, quality/detection/alignment/embedding pipeline 28.26 ms. The last timing excludes template decryption, threshold loading and a verification decision, so it is not full verification latency. Decode/camera capture are excluded. Timing values describe this laptop/input/session and are not accuracy evidence.

One image per person is insufficient for enrollment; the two screenshots also fail blur QC. No calibrated matches, real FMR/FNMR or webcam success are claimed. **NOT READY FOR UNSUPERVISED ATTENDANCE.**

## Latest batch outside Jupyter and privacy checks

This section reads the executed ten-image retest artifact. The initial three-image experiment above is retained as history. User-supplied labels are represented by pseudonyms; no portraits or vectors are displayed. The user confirmed all ten captures are from one session. Enrollment uses photos 1/2/4 and 7/8/9; probes use 3/5 and 6/10, selected before score inspection. The encrypted enrollment smoke used a process-only key and removed all templates afterward.

These results establish implementation behavior, not a calibrated matching decision, independent-session accuracy, liveness, FMR/FNMR reliability or production readiness.

In [ ]:
batch = json.loads((REPORTS/'batch02_results.json').read_text())
print(json.dumps({'runner':'scripts/phase2_labeled_batch_smoke.py (executed in a fresh interpreter)',
    'status':batch['status'],'private_images_outside_project':batch['private_images_outside_project'],
    'embeddings_persisted_after_smoke':batch['embeddings_persisted_after_smoke'],
    'keys_persisted':batch['keys_persisted'],
    'photos_or_vectors_uploaded_to_github':batch['photos_or_vectors_uploaded_to_github']},indent=2))
assert batch['private_images_outside_project'] and not batch['photos_or_vectors_uploaded_to_github']

## Latest single-subject cross-session diagnostic

This section reads `batch03_results.json`, produced by an actual fresh-process GPU run. The user supplied one subject's ten photos and confirmed every photo came from a separate session; exact dates were not supplied. Enrollment photos 1/2/7 and probe photos 3/4/5/6/8/9/10 were fixed before inspecting scores. Photos 6/10 fail the unchanged size/clipping gate. No rejected image is embedded.

The three-image encrypted enrollment, duplicate-ID rejection and deletion passed. Five valid probes have diagnostic genuine scores 0.529–0.835; ten comparisons with explicitly labeled prior controls have scores 0.022–0.268. Session independence is user-confirmed for the new subject; cross-subject chronology is unconfirmed. Comparisons share subjects/templates, so the counts are dependent evidence, not independent population trials. All operational verification attempts refuse with THRESHOLD_NOT_CALIBRATED. No production threshold, FMR/FNMR, liveness or attendance claim is issued.

Names, original images and vectors remain out of public notebook outputs. Test templates and the process-only key were removed. Earlier cohorts above are preserved as history.

In [ ]:
latest = json.loads((REPORTS/'batch03_results.json').read_text())
print(json.dumps({'runner':'scripts/phase2_cross_session_smoke.py (fresh interpreter)','status':latest['status'],'private_images_outside_project':latest['private_images_outside_project'],'embeddings_persisted_after_smoke':latest['embeddings_persisted_after_smoke'],'keys_persisted':latest['keys_persisted'],'photos_or_vectors_uploaded_to_github':latest['photos_or_vectors_uploaded_to_github']},indent=2))